In [3]:
import os
import torch
import torch.optim as optim
import numpy as np
import pandas as pd

from data_sampler import data_aggregator
from PINN_module_flavours import NavierStokesPINN
from Lid_cavity_Physics_geo import LidDrivenCavity

In [4]:
data_folder = './data'
data_aggregator(mode='h')


                    DATA AGGREGATOR & SAMPLER: HELP MENU
Usage:
  data_aggregator(folder_path, sample_ratio=0.1, mode='r', inspection=False, random_state=42)

Parameters:
  folder_path   : Path to folder containing target .csv simulation files.
  sample_ratio  : Float between (0.0, 1.0] specifying sampling fraction (e.g. 0.15 = 15%).
  inspection    : Boolean flag. If True, inspects file metadata and returns a summary dict.
  random_state  : Integer seed for reproducible random selection.
  mode          : Single-character operation flag:
                  'h' - Display this instruction menu and return None.
                  'r' - Random uniform sampling (shuffles rows, preserves distributions).
                  's' - Systematic stride sampling (step = 1/ratio, maintains spatial order).
                  'f' - Head slice (extracts first sample_ratio fraction of points).
                  'l' - Tail slice (extracts last sample_ratio fraction of points).



In [5]:
reference_df = data_aggregator(
        folder_path=data_folder, 
        sample_ratio=1.0, 
        mode='r', 
        inspection=False
    )
print(reference_df)

Discovered 6 CSV files. Reading and accumulating...

Concatenation Complete.
Master Dataset Shape: (64848, 8) (64,848 total points)
Mode applied: Random Uniform
Sampled Dataset Shape: (64848, 8) (64,848 points at 100.0%)

       TimeStep  Points:0  Points:1  Points:2      U:0     U:1  U:2       p
0             1     0.062     0.052      0.01 -0.19600 -0.1050    0  0.0694
1             1     0.066     0.074      0.01 -0.08220 -0.1570    0  0.1580
2             4     0.076     0.036      0.01 -0.09190 -0.1020    0  0.0891
3             3     0.056     0.036      0.01 -0.16200 -0.0315    0  0.0436
4             2     0.092     0.062      0.00 -0.03530 -0.1930    0  0.2650
...         ...       ...       ...       ...      ...     ...  ...     ...
64843         5     0.012     0.034      0.01 -0.02790  0.0699    0 -0.0146
64844         3     0.004     0.052      0.00 -0.00619  0.0687    0 -0.0802
64845         0     0.088     0.032      0.00  0.00000  0.0000    0  0.0000
64846         1   

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NavierStokesPINN(in_features=3, num_layers=6, num_hidden=60, out_features=3).to(device)

In [ ]:
cavity = LDC_solver_weightannealing(Re=100.0, U_lid=1.0, L=1.0, device=device)



                    LID-DRIVEN CAVITY: SYSTEM CONFIGURATION
1. DATA SLICING SPECIFICATION:
   Expected sequence : [t, x, y, z, u, v, w, p]
   Retained fields   : [t (0), x (1), y (2), u (4), v (5), p (7)]
   Omitted fields    : z (Col 3), w (Col 6)

2. DOMAINS & PHYSICAL PARAMETERS:
   - Reynolds Number (Re) : 100.0 (1/Re = 1.00000e-02)
   - Spatial Span         : x in [0.0, 1.0], y in [0.0, 1.0]
   - Time Span            : t in [0.0, 1.0]
   - Lid Velocity (U_lid) : 1.0 (top wall y = 1.0)
   - Collocation Points   : PDE=2500, Wall=250/wall, IC=1000

3. GOVERNING EQUATIONS (Non-Dimensional Incompressible Navier-Stokes):
   - Continuity : ∂u/∂x + ∂v/∂y = 0
   - x-Momentum : ∂u/∂t + u(∂u/∂x) + v(∂u/∂y) + ∂p/∂x - (1/Re)(∂²u/∂x² + ∂²u/∂y²) = 0
   - y-Momentum : ∂v/∂t + u(∂v/∂x) + v(∂v/∂y) + ∂p/∂y - (1/Re)(∂²v/∂x² + ∂²v/∂y²) = 0

4. BOUNDARY & INITIAL CONDITIONS:
   - Top Wall (y = L)    : u = 1.0, v = 0, ∂p/∂y = 0
   - Bottom Wall (y = 0) : u = 0, v = 0, ∂p/∂y = 0
   - Left Wall (x = 0)  

In [ ]:

print(type(cavity))
loss_dict,loss = cavity.LDC_solver(model=model,data_df=reference_df)
print(lossloss,_dict)

<class 'Lid_cavity_Physics_geo.LidDrivenCavity'>
{'total': tensor(1.6015, grad_fn=<AddBackward0>), 'pde': tensor(6.2717e-08, grad_fn=<AddBackward0>), 'bc': tensor(0.8812, grad_fn=<AddBackward0>), 'ic': tensor(0.0101, grad_fn=<AddBackward0>), 'data': tensor(0.7102, grad_fn=<AddBackward0>)} tensor(1.6015, grad_fn=<AddBackward0>)


In [17]:
# Target convergence threshold
optimizer = optim.Adam(model.parameters(), lr=1e-2)
target_error = 1e-3
epoch = 0

model.train()
print(f"Beginning optimization until total loss < {target_error:.1e}...\n")

while True:
    optimizer.zero_grad()

    # Unified forward pass: slices data, computes autograd residuals, BCs, ICs, and data loss
    total_loss, losses = cavity.LDC_solver(model, reference_df)

    # Backpropagation and optimization step
    total_loss.backward()
    optimizer.step()

    # Logging progress
    if epoch % 100 == 0:
        print(
            f"Epoch {epoch:5d} | "
            f"Total: {total_loss.item():.5e} | "
            f"PDE: {losses['pde'].item():.4e} | "
            f"BC: {losses['bc'].item():.4e} | "
            f"IC: {losses['ic'].item():.4e} | "
            f"Data: {losses['data'].item():.4e}"
        )

    # Convergence check
    if total_loss.item() < target_error:
        print(f"\nConverged at Epoch {epoch}!")
        print(f"Final Total Loss: {total_loss.item():.6e} < {target_error:.1e}")
        break

    epoch += 1

Beginning optimization until total loss < 1.0e-03...

Epoch     0 | Total: 1.60149e+00 | PDE: 6.1449e-08 | BC: 8.8122e-01 | IC: 1.0110e-02 | Data: 7.1015e-01
Epoch   100 | Total: 8.44700e-01 | PDE: 4.3284e-03 | BC: 1.1532e-01 | IC: 1.9337e-02 | Data: 7.0571e-01
Epoch   200 | Total: 7.96285e-01 | PDE: 1.3149e-02 | BC: 6.7310e-02 | IC: 1.1075e-02 | Data: 7.0475e-01
Epoch   300 | Total: 7.94401e-01 | PDE: 1.2687e-02 | BC: 6.4766e-02 | IC: 1.2229e-02 | Data: 7.0472e-01
Epoch   400 | Total: 7.88956e-01 | PDE: 9.6436e-03 | BC: 5.9573e-02 | IC: 1.5273e-02 | Data: 7.0447e-01
Epoch   500 | Total: 7.85575e-01 | PDE: 8.3186e-03 | BC: 5.8948e-02 | IC: 1.3972e-02 | Data: 7.0434e-01
Epoch   600 | Total: 7.79093e-01 | PDE: 7.4776e-03 | BC: 5.1779e-02 | IC: 1.5460e-02 | Data: 7.0438e-01
Epoch   700 | Total: 8.00795e-01 | PDE: 6.2075e-03 | BC: 7.6988e-02 | IC: 1.3206e-02 | Data: 7.0439e-01
Epoch   800 | Total: 7.83210e-01 | PDE: 5.0275e-03 | BC: 6.4499e-02 | IC: 9.3039e-03 | Data: 7.0438e-01
Epoch   90

KeyboardInterrupt: 